# MindSync — 8-fold speaker cross-validation (this kernel: folds [0, 1, 2, 3, 4, 5, 6, 7])
Speaker groups G_k = actors 3k+1..3k+3 (k = 0..7). Fold k: test = G_k, validation = G_(k-1 mod 8), train = the other 18 actors.
Fold 7 reproduces the original split (test 22–24, val 19–21). Per fold: Stage 2 v3 audio (seed 42) → frame features →
out-of-sample fusion (GoEmotions-validation 80% × first two val actors; selection on the other 20% × third val actor) →
test on GoEmotions test × fold test actors → JSD, rule-based and learned alerts → RAVDESS same-utterance test.
All analysis choices below were fixed before any CV result was seen.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.45.2', 'datasets==2.21.0', 'huggingface_hub==0.25.2'], check=True)

import os, gc, json, math, time, random, pathlib, urllib.request, zipfile, collections, traceback, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore', category=FutureWarning)

T_START = time.time()
WORK = pathlib.Path('/kaggle/working'); WORK.mkdir(parents=True, exist_ok=True)
DATA_DIR = pathlib.Path('/tmp/ravdess'); DATA_DIR.mkdir(parents=True, exist_ok=True)   # outside /kaggle/working -> not saved as output
RESULTS_PATH = WORK / 'cv_results_all.json'
BEST_CKPT = WORK / 'audio_v3_best.pt'

assert torch.cuda.is_available(), 'No GPU - enable the T4 accelerator'
DEVICE = torch.device('cuda')
AMP = DEVICE.type == 'cuda'

import transformers, librosa, sklearn
transformers.logging.set_verbosity_error()
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| librosa', librosa.__version__, '| sklearn', sklearn.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', flush=True)
FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
ARR_DIR = WORK / 'cv_arrays'; ARR_DIR.mkdir(exist_ok=True)

In [ ]:
"""Shared Stage 2 v3 audio model (wav2vec2-Large-960h, same checkpoint as v1).

Fixes relative to v1 (which mean-pooled the LAST layer over zero-padding):
  1. learned softmax-weighted sum over all 25 hidden states (each LayerNorm-ed first)
  2. mean+std pooling over VALID frames only (frame mask from true input lengths)
  3. (training side) waveform augmentation + light SpecAugment

Both the Stage 2 v3 training notebook and the Stage 3 v3 notebook paste this
class VERBATIM so the saved state_dict always loads.
"""
import torch
import torch.nn as nn
from transformers import Wav2Vec2Config, Wav2Vec2Model

AUDIO_MODEL_NAME = 'facebook/wav2vec2-large-960h'


class AudioModelV3(nn.Module):
    def __init__(self, model_name=AUDIO_MODEL_NAME, num_classes=4, freeze_layers=8,
                 dropout=0.1, spec_augment=True):
        super().__init__()
        cfg = Wav2Vec2Config.from_pretrained(model_name)
        cfg.output_hidden_states = True
        cfg.apply_spec_augment = spec_augment
        cfg.mask_time_prob = 0.05
        cfg.mask_time_length = 5
        cfg.mask_feature_prob = 0.0
        self.enc = Wav2Vec2Model.from_pretrained(model_name, config=cfg)
        self.enc.freeze_feature_encoder()
        for layer in self.enc.encoder.layers[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False
        # 'layer' norm models accept attention_mask; 'group' norm models (e.g. -960h) must not get one
        self.use_attention_mask = (cfg.feat_extract_norm == 'layer')
        H = cfg.hidden_size
        L = cfg.num_hidden_layers + 1
        self.layer_w = nn.Parameter(torch.zeros(L))
        self.layer_norm = nn.LayerNorm(H)
        self.proj = nn.Sequential(nn.LayerNorm(2 * H), nn.Linear(2 * H, H))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(H, H // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(H // 2, num_classes))

    def frames(self, input_values, lengths):
        """input_values: (B, S) zero-padded waveforms; lengths: (B,) true sample counts.
        Returns frame features (B, T, H) and a bool frame mask (B, T), True = valid."""
        if self.use_attention_mask:
            am = (torch.arange(input_values.shape[1], device=input_values.device)[None, :]
                  < lengths[:, None]).long()
            out = self.enc(input_values=input_values, attention_mask=am, output_hidden_states=True)
        else:
            out = self.enc(input_values=input_values, output_hidden_states=True)
        hs = torch.stack([self.layer_norm(h) for h in out.hidden_states], 0)   # (L, B, T, H)
        w = torch.softmax(self.layer_w, 0)
        f = (w[:, None, None, None] * hs).sum(0)                               # (B, T, H)
        flen = self.enc._get_feat_extract_output_lengths(lengths).clamp(min=1)
        mask = torch.arange(f.shape[1], device=f.device)[None, :] < flen[:, None]
        return f, mask

    @staticmethod
    def pool(f, mask):
        m = mask.unsqueeze(-1).to(f.dtype)
        n = m.sum(1).clamp_min(1.0)
        mean = (f * m).sum(1) / n
        var = (((f - mean[:, None, :]) ** 2) * m).sum(1) / n
        return torch.cat([mean, var.clamp_min(1e-6).sqrt()], -1)

    def forward(self, input_values, lengths):
        f, mask = self.frames(input_values, lengths)
        emb = self.proj(self.pool(f, mask))
        return {'embedding': emb, 'logits': self.head(emb)}

In [ ]:
ZIP = DATA_DIR / 'ravdess.zip'
AUDIO_ROOT = DATA_DIR / 'extracted'
if not AUDIO_ROOT.exists() or len(list(AUDIO_ROOT.rglob('*.wav'))) < 1440:
    if not ZIP.exists() or ZIP.stat().st_size < 100_000_000:
        print('Downloading RAVDESS speech (~200 MB) from Zenodo ...', flush=True)
        urllib.request.urlretrieve('https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip', ZIP)
    AUDIO_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(AUDIO_ROOT)
    ZIP.unlink()
wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
print('RAVDESS wav files:', len(wavs), flush=True)

In [ ]:
CLUSTER_NAMES = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
C2I = {c: i for i, c in enumerate(CLUSTER_NAMES)}
RAVDESS_TO_CLUSTER = {1: 'Ambiguity', 2: 'Resilience', 3: 'Resilience', 4: 'Distress',
                      5: 'Aggression', 6: 'Distress', 7: 'Aggression', 8: 'Ambiguity'}
by_name = {}
for p in wavs:
    if len(p.stem.split('-')) == 7:
        by_name.setdefault(p.name, p)
items, EMO, STMT = [], [], []                     # items: (path, cluster_idx, actor)
for nm, p in sorted(by_name.items()):
    parts = p.stem.split('-')
    emo, stmt, actor = int(parts[2]), int(parts[4]), int(parts[6])
    items.append((str(p), C2I[RAVDESS_TO_CLUSTER[emo]], actor)); EMO.append(emo); STMT.append(stmt)
assert len(items) == 1440
EMO, STMT = np.array(EMO), np.array(STMT)
GROUP = lambda k: {3 * k + 1, 3 * k + 2, 3 * k + 3}
def fold_splits(k):
    te, va = GROUP(k), GROUP((k - 1) % 8)
    return {'train': [s for s in items if s[2] not in te | va], 'val': [s for s in items if s[2] in va],
            'test': [s for s in items if s[2] in te]}
for k in range(8):
    sp = fold_splits(k); A = {kk: {s[2] for s in v} for kk, v in sp.items()}
    assert [len(sp[x]) for x in ('train', 'val', 'test')] == [1080, 180, 180]
    assert not (A['train'] & A['val']) and not (A['train'] & A['test']) and not (A['val'] & A['test'])
SPLITS = fold_splits(7)                            # default (= original split) for the helper cells below
print('8 speaker-disjoint folds OK; fold 7 =', {k: sorted({s[2] for s in v}) for k, v in SPLITS.items() if k != 'train'})

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import Wav2Vec2FeatureExtractor

SR, MAX_SAMPLES, MIN_SAMPLES = 16_000, 80_000, 400      # 5 s crop; 400 samples = wav2vec2 receptive field
fe = Wav2Vec2FeatureExtractor.from_pretrained(AUDIO_MODEL_NAME)
print(f'feature extractor: do_normalize={fe.do_normalize} | padding_value={fe.padding_value} | '
      f'sampling_rate={fe.sampling_rate} | return_attention_mask={fe.return_attention_mask}')
assert fe.do_normalize and fe.sampling_rate == SR

t0 = time.time(); WAV = {}
for i, (path, _, _) in enumerate(items):
    w, _ = librosa.load(path, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    WAV[path] = w.astype(np.float32)
    if (i + 1) % 360 == 0:
        print(f'  decoded {i + 1}/{len(items)}', flush=True)
secs = np.array([len(w) for w in WAV.values()]) / SR
print(f'decoded in {time.time() - t0:.0f}s | trimmed length mean {secs.mean():.2f}s, min {secs.min():.2f}s, '
      f'max {secs.max():.2f}s | longer than 5 s: {(secs > 5).mean():.1%}', flush=True)


def crop(w, train):
    if len(w) <= MAX_SAMPLES:
        return w
    s = random.randint(0, len(w) - MAX_SAMPLES) if train else (len(w) - MAX_SAMPLES) // 2   # eval: CENTRE crop
    return w[s:s + MAX_SAMPLES]


def augment(w):
    speed = random.uniform(0.9, 1.1)                          # speed perturbation via resampling
    w = librosa.resample(w, orig_sr=SR, target_sr=int(round(SR / speed)))
    w = w * random.uniform(0.8, 1.2)                          # gain (kept as specified; the per-clip normalisation
                                                              #       below makes it almost a no-op)
    if random.random() < 0.5:                                 # additive white noise at 15-30 dB SNR
        snr_db = random.uniform(15.0, 30.0)
        rms = float(np.sqrt(np.mean(w ** 2))) + 1e-9
        w = w + np.random.normal(0.0, rms / 10 ** (snr_db / 20), len(w))
    return w.astype(np.float32)


class RavdessDS(Dataset):
    """(normalised waveform, label). Train: augment + random 5 s crop. Eval: centre 5 s crop."""
    def __init__(self, rows, train):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        path, y, _ = self.rows[i]
        w = WAV[path]
        if self.train:
            w = augment(w)      # python `random` / numpy are re-seeded per worker per epoch from the loader's generator
        w = crop(w, self.train)
        if len(w) < MIN_SAMPLES:
            w = np.pad(w, (0, MIN_SAMPLES - len(w)))
        x = fe(w, sampling_rate=SR, return_tensors='np')['input_values'][0].astype(np.float32)   # this clip only
        return x, y


def collate(batch):
    """Zero-pads to the longest clip. `lengths` = true sample counts (after crop + normalisation)."""
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    iv = torch.zeros(len(batch), int(lengths.max()), dtype=torch.float32)
    for i, (x, _) in enumerate(batch):
        iv[i, :len(x)] = torch.from_numpy(x)
    return {'input_values': iv, 'lengths': lengths,
            'labels': torch.tensor([y for _, y in batch], dtype=torch.long)}


_ds = RavdessDS(SPLITS['val'], False)
_b = collate([_ds[0], _ds[7]])
_L = _b['lengths'].tolist()
assert _b['input_values'].shape == (2, max(_L)) and max(_L) <= MAX_SAMPLES
for _i, _l in enumerate(_L):
    assert torch.all(_b['input_values'][_i, _l:] == 0)
print('collate check: input_values', tuple(_b['input_values'].shape), '| lengths', _L, '| labels', _b['labels'].tolist())

In [ ]:
SEEDS          = [42]               # one audio seed per fold (fold-to-fold spread replaces seed spread)
FREEZE_LAYERS  = 8            # AudioModelV3 default; stored in the checkpoint
BATCH, ACCUM   = 1, 16        # effective batch 16; batch 1 => no zero-padding ever reaches the -960h encoder
                              # (GroupNorm frontend, no attention_mask), so training matches eval and Stage 3 extraction
EVAL_BATCH     = 1            # val/test one clip at a time -> no padding reaches the -960h encoder (it gets no attention_mask)
NUM_WORKERS    = 2
LR_BACKBONE    = 1e-5         # unfrozen wav2vec2 parameters
LR_NEW         = 3e-4         # layer_w, layer_norm, proj, head
WEIGHT_DECAY   = 0.01
WARMUP_FRAC    = 0.10
LABEL_SMOOTH   = 0.05
GRAD_CLIP      = 1.0
MAX_EPOCHS     = 20
PATIENCE       = 5            # early stopping on val macro-F1
TIME_BUDGET_H  = 11.3         # Kaggle hard limit is 12 h: no new epoch / seed is started after this
V1_TEST        = {'acc': 0.4056, 'f1': 0.3996}

CONFIG = dict(model_name=AUDIO_MODEL_NAME, model_class='AudioModelV3', freeze_layers=FREEZE_LAYERS, seeds=SEEDS,
              batch=BATCH, grad_accum=ACCUM, eval_batch=EVAL_BATCH, lr_backbone=LR_BACKBONE, lr_new=LR_NEW,
              weight_decay=WEIGHT_DECAY, warmup_frac=WARMUP_FRAC, label_smoothing=LABEL_SMOOTH, grad_clip=GRAD_CLIP,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, amp=AMP, sampler='class-balanced WeightedRandomSampler',
              augmentation='speed U(0.9,1.1) via resampling, gain U(0.8,1.2), white noise SNR U(15,30) dB p=0.5; SpecAugment in encoder',
              preprocessing='librosa 16 kHz mono, trim top_db=30, 5 s crop (train random / eval centre), per-clip Wav2Vec2FeatureExtractor normalisation',
              split='speaker-independent: train actors 1-18 / val 19-21 / test 22-24',
              cluster_map={str(k): v for k, v in RAVDESS_TO_CLUSTER.items()}, cluster_names=CLUSTER_NAMES,
              selection='best epoch and saved seed chosen by VALIDATION macro-F1', v1_test=V1_TEST,
              versions={'torch': torch.__version__, 'transformers': transformers.__version__, 'librosa': librosa.__version__})
print(json.dumps(CONFIG, indent=1))

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix

LABELS = [0, 1, 2, 3]


def hours():
    return (time.time() - T_START) / 3600


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP)


def _json_default(o):
    return o.item() if hasattr(o, 'item') else str(o)


def make_loaders(seed):
    rows = SPLITS['train']
    counts = np.bincount([r[1] for r in rows], minlength=4)
    weights = torch.tensor([1.0 / counts[r[1]] for r in rows], dtype=torch.double)
    sampler = WeightedRandomSampler(weights, num_samples=len(rows), replacement=True,
                                    generator=torch.Generator().manual_seed(seed))
    kw = dict(num_workers=NUM_WORKERS, pin_memory=AMP, collate_fn=collate,
              multiprocessing_context='fork' if NUM_WORKERS > 0 else None)
    train = DataLoader(RavdessDS(rows, True), batch_size=BATCH, sampler=sampler,
                       generator=torch.Generator().manual_seed(seed + 10_000), **kw)
    val = DataLoader(RavdessDS(SPLITS['val'], False), batch_size=EVAL_BATCH, shuffle=False, **kw)
    test = DataLoader(RavdessDS(SPLITS['test'], False), batch_size=EVAL_BATCH, shuffle=False, **kw)
    return train, val, test


def forward_logits(model, b):
    return model(b['input_values'].to(DEVICE, non_blocking=True),
                 b['lengths'].to(DEVICE, non_blocking=True))['logits']


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for b in loader:
        with autocast():
            logits = forward_logits(model, b)
        P += logits.float().argmax(-1).cpu().tolist(); Y += b['labels'].tolist()
    P, Y = np.array(P), np.array(Y)
    return {'acc': float((P == Y).mean()),
            'f1': float(f1_score(Y, P, labels=LABELS, average='macro', zero_division=0)),
            'report': classification_report(Y, P, labels=LABELS, target_names=CLUSTER_NAMES, digits=4,
                                            output_dict=True, zero_division=0),
            'confusion': confusion_matrix(Y, P, labels=LABELS).tolist(),
            'n': int(len(Y))}


SANITY_ROWS = random.Random(0).sample(SPLITS['val'], 32)     # fixed val subset, same for every seed


@torch.no_grad()
def initial_loss_check(model):
    loader = DataLoader(RavdessDS(SANITY_ROWS, False), batch_size=BATCH, shuffle=False, collate_fn=collate)
    model.eval(); tot, n, stds = 0.0, 0, []
    for b in loader:
        with autocast():
            logits = forward_logits(model, b)
        tot += F.cross_entropy(logits.float(), b['labels'].to(DEVICE), reduction='sum').item()
        n += len(b['labels']); stds.append(logits.float().std().item())
    loss = tot / n
    status = 'PASS' if abs(loss - math.log(4)) < 0.3 else 'FAIL'
    print(f'  initial-loss sanity check: {loss:.4f} (expected ~ ln 4 = {math.log(4):.4f}; '
          f'logit std {np.mean(stds):.3f}) -> {status}', flush=True)
    return float(loss), status


def run_seed(seed):
    set_seed(seed)
    model = AudioModelV3(model_name=AUDIO_MODEL_NAME, num_classes=4, freeze_layers=FREEZE_LAYERS).to(DEVICE)
    new_params = [model.layer_w, *model.layer_norm.parameters(), *model.proj.parameters(), *model.head.parameters()]
    backbone_params = [p for p in model.enc.parameters() if p.requires_grad]
    trainable = [p for p in model.parameters() if p.requires_grad]
    assert not ({id(p) for p in new_params} & {id(p) for p in backbone_params})
    assert len(new_params) + len(backbone_params) == len(trainable)
    print(f'  params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M total | '
          f'backbone trainable {sum(p.numel() for p in backbone_params) / 1e6:.1f} M (lr {LR_BACKBONE}) | '
          f'new {sum(p.numel() for p in new_params) / 1e6:.2f} M (lr {LR_NEW})', flush=True)

    train_loader, val_loader, test_loader = make_loaders(seed)
    init_loss, init_status = initial_loss_check(model)

    optim = AdamW([{'params': backbone_params, 'lr': LR_BACKBONE},
                   {'params': new_params, 'lr': LR_NEW}], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_loader) / ACCUM)
    total_steps = steps_per_epoch * MAX_EPOCHS
    sched = get_cosine_schedule_with_warmup(optim, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    print(f'  {len(train_loader)} batches x {BATCH} -> {steps_per_epoch} optimiser steps/epoch, '
          f'{total_steps} max steps, warm-up {int(WARMUP_FRAC * total_steps)}', flush=True)
    if AMP:
        torch.cuda.reset_peak_memory_stats()

    best_f1, best_epoch, bad, best_state, history, stop_reason = -1.0, 0, 0, None, [], 'max_epochs'
    t_train = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        if hours() > TIME_BUDGET_H:
            stop_reason = 'time_budget'
            print(f'  time budget ({TIME_BUDGET_H} h) reached - stopping this seed', flush=True)
            break
        model.train(); t0 = time.time()
        loss_sum, correct, seen, nonfinite = 0.0, 0, 0, 0
        optim.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            y = b['labels'].to(DEVICE, non_blocking=True)
            with autocast():
                logits = forward_logits(model, b)
            loss = crit(logits.float(), y)
            scaler.scale(loss / ACCUM).backward()
            if (i + 1) % ACCUM == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
                scaler.step(optim); scaler.update()
                optim.zero_grad(set_to_none=True)
                sched.step()
            if torch.isfinite(loss):
                loss_sum += loss.item() * len(y)
            else:
                nonfinite += 1
            seen += len(y); correct += int((logits.argmax(-1) == y).sum())
            if (i + 1) % 90 == 0:
                print(f'    ep {ep} step {i + 1}/{len(train_loader)} | running loss {loss_sum / max(seen, 1):.4f}', flush=True)

        v = evaluate(model, val_loader)
        lrs = sched.get_last_lr()
        history.append({'epoch': ep, 'train_loss': loss_sum / max(seen, 1), 'train_acc': correct / max(seen, 1),
                        'val_acc': v['acc'], 'val_f1': v['f1'], 'lr_backbone': lrs[0], 'lr_new': lrs[1],
                        'nonfinite_losses': nonfinite, 'epoch_sec': time.time() - t0})
        mark = ''
        if v['f1'] > best_f1:
            best_f1, best_epoch, bad = v['f1'], ep, 0
            best_state = {k: t.detach().to('cpu', copy=True) for k, t in model.state_dict().items()}
            mark = '  * best'
        else:
            bad += 1
        print(f'  seed {seed} | epoch {ep:2d} | train loss {history[-1]["train_loss"]:.4f} acc {history[-1]["train_acc"]:.4f} | '
              f'val acc {v["acc"]:.4f} macro-F1 {v["f1"]:.4f} | {time.time() - t0:.0f}s | elapsed {hours():.2f} h{mark}', flush=True)
        if bad >= PATIENCE:
            stop_reason = 'early_stopping'
            print(f'  early stop: {PATIENCE} epochs without val macro-F1 improvement', flush=True)
            break

    if best_state is None:
        raise RuntimeError('no epoch completed (time budget)')
    model.load_state_dict(best_state)
    val_m, test_m = evaluate(model, val_loader), evaluate(model, test_loader)
    lw = torch.softmax(model.layer_w.detach().float().cpu(), 0)
    run = {'seed': seed, 'best_epoch': best_epoch, 'epochs_run': len(history), 'stop_reason': stop_reason,
           'initial_loss': init_loss, 'initial_loss_check': init_status,
           'best_val_f1_during_training': float(best_f1),
           'val': val_m, 'test': test_m,
           'layer_weights': lw.tolist(),
           'top5_layers': [[int(i), float(lw[i])] for i in torch.argsort(lw, descending=True)[:5]],
           'history': history, 'train_minutes': (time.time() - t_train) / 60,
           'peak_gpu_mem_gb': torch.cuda.max_memory_allocated() / 1e9 if AMP else None}
    print(f'  seed {seed} best epoch {best_epoch} -> val acc {val_m["acc"]:.4f} F1 {val_m["f1"]:.4f} | '
          f'test acc {test_m["acc"]:.4f} F1 {test_m["f1"]:.4f}   (v1 test: acc {V1_TEST["acc"]} F1 {V1_TEST["f1"]})', flush=True)
    print('  top-5 layers (softmax layer_w):', run['top5_layers'], flush=True)
    return run, best_state


def save_best(state, run):
    tmp = WORK / (BEST_CKPT.name + '.tmp')
    torch.save({'model_state_dict': state, 'seed': run['seed'], 'val_f1': run['val']['f1'],
                'test_acc': run['test']['acc'], 'test_f1': run['test']['f1'],
                'cluster_names': CLUSTER_NAMES, 'model_name': AUDIO_MODEL_NAME,
                'freeze_layers': FREEZE_LAYERS}, tmp)
    os.replace(tmp, BEST_CKPT)                  # atomic: only one checkpoint ever remains on disk


def write_results():
    tmp = WORK / (RESULTS_PATH.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(RESULTS, f, indent=1, default=_json_default)
    os.replace(tmp, RESULTS_PATH)

## Text features (Stage 1, fp32): GoEmotions validation + test, and the two RAVDESS transcripts

In [ ]:
from transformers import RobertaModel, AutoTokenizer
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
device = DEVICE

class MindSyncTextModel(nn.Module):
    def __init__(self, model_name='roberta-large', num_classes=4, dropout=0.1):
        super().__init__()
        self.encoder = nn.Module(); self.encoder.encoder = RobertaModel.from_pretrained(model_name)
        self.encoder.hidden_size = self.encoder.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Module(); h = self.encoder.hidden_size
        self.classifier.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, h // 2), nn.GELU(),
                                                   nn.Dropout(dropout), nn.Linear(h // 2, num_classes))
sd = torch.load(hf_hub_download('Ubaida1/mindsync-text-model', 'best_text_model.pt'), map_location='cpu', weights_only=False)
sd = sd.get('model_state_dict', sd) if isinstance(sd, dict) else sd
text_model = MindSyncTextModel(); text_model.load_state_dict(sd); text_model.to(device).eval()
for p_ in text_model.parameters(): p_.requires_grad = False
del sd

EMOTION_TO_CLUSTER = {
    'grief': 'Distress', 'nervousness': 'Distress', 'fear': 'Distress', 'sadness': 'Distress', 'remorse': 'Distress',
    'joy': 'Resilience', 'admiration': 'Resilience', 'excitement': 'Resilience', 'relief': 'Resilience',
    'amusement': 'Resilience', 'approval': 'Resilience', 'curiosity': 'Resilience', 'desire': 'Resilience',
    'gratitude': 'Resilience', 'love': 'Resilience', 'optimism': 'Resilience', 'pride': 'Resilience',
    'realization': 'Resilience',
    'anger': 'Aggression', 'annoyance': 'Aggression', 'disgust': 'Aggression', 'disapproval': 'Aggression',
    'embarrassment': 'Aggression',
    'confusion': 'Ambiguity', 'disappointment': 'Ambiguity', 'surprise': 'Ambiguity', 'caring': 'Ambiguity',
    'neutral': 'Ambiguity'}
tok = AutoTokenizer.from_pretrained('roberta-large'); NTOK = 64
def encode_texts(texts):
    cls, toks, tmask = [], [], []
    for i in range(0, len(texts), 32):
        enc = tok(texts[i:i + 32], padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad():
            h = text_model.encoder.encoder(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device)).last_hidden_state
        cls.append(h[:, 0].float().cpu()); toks.append(h[:, :NTOK].half().cpu()); tmask.append(enc['attention_mask'][:, :NTOK].bool())
    emb = torch.cat(cls)
    with torch.no_grad():
        probs = F.softmax(text_model.classifier.classifier(emb.to(device)).float(), -1).cpu()
    return {'emb': emb, 'tok': torch.cat(toks), 'tmask': torch.cat(tmask), 'probs': probs}

ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
names = ds['train'].features['labels'].feature.names
to_c = lambda ids: C2I[EMOTION_TO_CLUSTER.get(names[ids[0]], 'Ambiguity')] if ids else None
T = {}
for split in ['validation', 'test']:
    its = [(it['text'], to_c(it['labels'])) for it in ds[split]]; its = [x for x in its if x[1] is not None]
    T[split] = encode_texts([x[0] for x in its]); T[split]['cluster'] = torch.tensor([x[1] for x in its]); T[split]['texts'] = [x[0] for x in its]
    print(split, len(its), flush=True)
SENTENCES = ['Kids are talking by the door', 'Dogs are sitting by the door']      # RAVDESS statements 01 / 02
T['sent'] = encode_texts(SENTENCES); T['sent']['cluster'] = torch.tensor([C2I['Ambiguity']] * 2); T['sent']['texts'] = SENTENCES
yt_ = T['test']['cluster']; print(f"text test acc {accuracy_score(yt_, T['test']['probs'].argmax(-1)):.4f} (expect 0.7076)")
print('RAVDESS transcripts -> text posterior:', {s: [round(float(v), 3) for v in p] for s, p in zip(SENTENCES, T['sent']['probs'])})
text_model.cpu(); del text_model; torch.cuda.empty_cache()

## Fusion models, pair construction, metrics

In [ ]:
def mpool(x, m):
    mm = m.unsqueeze(-1).float(); return (x * mm).sum(1) / mm.sum(1).clamp_min(1.0)

class PooledCMAF(nn.Module):
    def __init__(self, d=256, heads=4, dropout=0.1, aux=True):
        super().__init__(); self.aux = aux
        self.tp, self.ap = nn.Linear(1024, d), nn.Linear(1024, d)
        self.t2a = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.nt, self.na = nn.LayerNorm(d), nn.LayerNorm(d)
        self.fuse = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 4))
        self.ht, self.ha = nn.Linear(d, 4), nn.Linear(d, 4)
    def forward(self, b):
        t, a = self.tp(b['cls']).unsqueeze(1), self.ap(b['aemb']).unsqueeze(1)
        ta, _ = self.t2a(t, a, a); at, _ = self.a2t(a, t, t)
        t, a = self.nt(t + ta).squeeze(1), self.na(a + at).squeeze(1)
        return {'fused_logits': self.fuse(torch.cat([t, a], -1)), 'text_logits': self.ht(t), 'audio_logits': self.ha(a)}

class SeqCMAF(nn.Module):
    def __init__(self, d=256, heads=4, dropout=0.1, aux=True):
        super().__init__(); self.aux = aux
        self.tp, self.ap = nn.Linear(1024, d), nn.Linear(1024, d)
        self.t2a = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.nt, self.na = nn.LayerNorm(d), nn.LayerNorm(d)
        self.fuse = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 4))
        self.ht, self.ha = nn.Linear(d, 4), nn.Linear(d, 4)
    def forward(self, b):
        t, a = self.tp(b['tok']), self.ap(b['fr'])
        ta, _ = self.t2a(t, a, a, key_padding_mask=~b['fm'], need_weights=False)
        at, _ = self.a2t(a, t, t, key_padding_mask=~b['tm'], need_weights=False)
        t, a = self.nt(t + ta), self.na(a + at)
        tp, ap = mpool(t, b['tm']), mpool(a, b['fm'])
        return {'fused_logits': self.fuse(torch.cat([tp, ap], -1)), 'text_logits': self.ht(tp), 'audio_logits': self.ha(ap)}

def build_pairs(tsplit, text_idx, audio_mask, seed, congruent=True):
    r = random.Random(seed)
    pool = {c: [i for i in np.nonzero(audio_mask)[0].tolist() if A_CL[i] == c] for c in range(4)}
    labels = T[tsplit]['cluster']; out = []
    for ti in text_idx:
        c = int(labels[ti]); ac = c if congruent else r.choice([k for k in range(4) if k != c])
        out.append((ti, r.choice(pool[ac]), c))
    return {'ts': tsplit, 'pairs': out}

def fetch(ps, idx):
    ts = T[ps['ts']]; pairs = ps['pairs']
    ti = torch.tensor([pairs[i][0] for i in idx]); ai = torch.tensor([pairs[i][1] for i in idx])
    return {'cls': ts['emb'][ti].to(device), 'tok': ts['tok'][ti].to(device).float(), 'tm': ts['tmask'][ti].to(device),
            'aemb': AEMB[ai].to(device), 'fr': AF[ai].to(device).float(), 'fm': AM[ai].to(device),
            'y': torch.tensor([pairs[i][2] for i in idx], device=device)}

def stream_probs(ps):
    ts = T[ps['ts']]; pairs = ps['pairs']
    ti = torch.tensor([p[0] for p in pairs]); ai = torch.tensor([p[1] for p in pairs])
    return ts['probs'][ti], APROB[ai], torch.tensor([p[2] for p in pairs])

@torch.no_grad()
def predict(model, ps, bs=256):
    model.eval(); outs = {'fused_logits': [], 'text_logits': [], 'audio_logits': []}; n = len(ps['pairs'])
    for i in range(0, n, bs):
        o = model(fetch(ps, range(i, min(i + bs, n))))
        for k in outs: outs[k].append(o[k].float().cpu())
    return {k: torch.cat(v) for k, v in outs.items()}

def train_fusion(cls_, seed, tr, va, epochs=10, lr=3e-4, alpha=0.3, bs=64):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = cls_().to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    y = torch.tensor([p[2] for p in tr['pairs']]); yv = np.array([p[2] for p in va['pairs']])
    w = (1.0 / torch.bincount(y, minlength=4).float())[y]; g = torch.Generator().manual_seed(seed)
    best, best_state = -1, None
    for ep in range(epochs):
        model.train(); idx = torch.multinomial(w, len(y), replacement=True, generator=g).tolist()
        for i in range(0, len(idx), bs):
            b = fetch(tr, idx[i:i + bs]); out = model(b)
            loss = F.cross_entropy(out['fused_logits'], b['y']) + alpha * (F.cross_entropy(out['text_logits'], b['y']) + F.cross_entropy(out['audio_logits'], b['y']))
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        vf1 = f1_score(yv, predict(model, va)['fused_logits'].argmax(-1).numpy(), average='macro')
        if vf1 > best: best, best_state = vf1, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); return model, float(best)

def jsd(p, q, eps=1e-8):
    m = 0.5 * (p + q); kl = lambda x: (x * (x.clamp_min(eps).log2() - m.clamp_min(eps).log2())).sum(-1)
    return 0.5 * (kl(p) + kl(q))
def ent(p): return -(p.clamp_min(1e-8) * p.clamp_min(1e-8).log()).sum(-1)
def alert_features(ps, model):
    pt, pa, _ = stream_probs(ps); o = predict(model, ps)
    aux = jsd(F.softmax(o['text_logits'], -1), F.softmax(o['audio_logits'], -1))
    X = torch.stack([jsd(pt, pa), pt.max(-1).values, pa.max(-1).values, ent(pt), ent(pa),
                     (pt.argmax(-1) != pa.argmax(-1)).float(), aux], 1).numpy()
    return X       # columns: jsd, max_pt, max_pa, ent_t, ent_a, mismatch, aux_jsd
def rates(score, thr, y):  # y: 1 = incongruent
    a = score >= thr; return {'detect': float(a[y == 1].mean()), 'false_alarm': float(a[y == 0].mean())}
def youden(score, y):
    best_t, best_j = 0.5, -9
    for t_ in np.unique(score):
        j = (score[y == 1] >= t_).mean() - (score[y == 0] >= t_).mean()
        if j > best_j: best_j, best_t = j, float(t_)
    return best_t
print('fusion/alert helpers ready', flush=True)

## Cross-validation loop

In [ ]:
def write_cv():
    tmp = WORK / (RESULTS_PATH.name + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f: json.dump(CV, f, indent=1, default=_json_default)
    os.replace(tmp, RESULTS_PATH)
CV = {'folds': {}, 'errors': []}
SEEDS_FUSION = [42, 43, 44]
perm = torch.randperm(len(T['validation']['cluster']), generator=torch.Generator().manual_seed(7)).tolist()
cut = int(0.8 * len(perm)); VAL_A, VAL_B = sorted(perm[:cut]), sorted(perm[cut:])
ALL = lambda ts: list(range(len(T[ts]['cluster'])))
A_CL = np.array([s[1] for s in items]); A_ACT = np.array([s[2] for s in items])
ds_all = RavdessDS(items, False)

for fold in FOLDS:
    if hours() > TIME_BUDGET_H:
        CV['errors'].append({'fold': fold, 'error': 'time budget'}); write_cv(); continue
    t_f = time.time(); print(f'\n================ fold {fold} ================', flush=True)
    try:
        SPLITS = fold_splits(fold)
        te_act, va_act = sorted(GROUP(fold)), sorted(GROUP((fold - 1) % 8))
        run, state = run_seed(42)
        am_ = AudioModelV3(model_name=AUDIO_MODEL_NAME, num_classes=4, freeze_layers=FREEZE_LAYERS).to(DEVICE)
        am_.load_state_dict(state); am_.eval(); del state
        frames, flen, emb_l, lg_l = [], [], [], []
        for k in range(len(items)):
            x, _ = ds_all[k]
            iv = torch.from_numpy(x)[None].to(DEVICE); ln = torch.tensor([x.shape[0]], device=DEVICE)
            with torch.no_grad(), autocast():
                f_, m_ = am_.frames(iv, ln); e_ = am_.proj(am_.pool(f_.float(), m_)); l_ = am_.head(e_)
            n_ = int(m_.sum()); frames.append(f_[0, :n_].half().cpu()); flen.append(n_)
            emb_l.append(e_[0].float().cpu()); lg_l.append(l_[0].float().cpu())
        del am_; torch.cuda.empty_cache(); gc.collect()
        TM = max(flen); AF = torch.zeros(len(items), TM, 1024, dtype=torch.float16); AM = torch.zeros(len(items), TM, dtype=torch.bool)
        for k, f_ in enumerate(frames): AF[k, :len(f_)] = f_; AM[k, :len(f_)] = True
        del frames
        AEMB = torch.stack(emb_l); APROB = F.softmax(torch.stack(lg_l), -1)
        te_mask = np.isin(A_ACT, te_act)
        a_test_acc = float((APROB[te_mask].argmax(-1).numpy() == A_CL[te_mask]).mean())
        assert abs(a_test_acc - run['test']['acc']) < 0.012, f'extraction acc {a_test_acc} vs training eval {run["test"]["acc"]}'

        TEST = build_pairs('test', ALL('test'), te_mask, 3)
        TR = build_pairs('validation', VAL_A, np.isin(A_ACT, va_act[:2]), 4)
        VA = build_pairs('validation', VAL_B, np.isin(A_ACT, va_act[2:]), 5)
        pt, pa, yt = stream_probs(TEST); late = (pt + pa).argmax(-1)
        res = {'test_actors': te_act, 'val_actors': va_act, 'audio_run': {k: run[k] for k in ('best_epoch', 'epochs_run', 'stop_reason', 'initial_loss_check')},
               'audio_val': {k: run['val'][k] for k in ('acc', 'f1')}, 'audio_test': {k: run['test'][k] for k in ('acc', 'f1', 'confusion')},
               'audio_test_report': run['test']['report'],
               'pairs_text_acc': float((pt.argmax(-1) == yt).float().mean()), 'pairs_audio_acc': float((pa.argmax(-1) == yt).float().mean()),
               'pairs_late_acc': float((late == yt).float().mean()), 'fusion': {}}
        preds = {'y': yt.numpy(), 'clip': np.array([p[1] for p in TEST['pairs']]), 'text': pt.argmax(-1).numpy(),
                 'audio': pa.argmax(-1).numpy(), 'late': late.numpy()}
        MODELS_F = {}
        for nm_, cls_ in [('seq', SeqCMAF), ('pooled', PooledCMAF)]:
            accs, f1s = [], []
            for s_ in SEEDS_FUSION:
                mdl, vf1 = train_fusion(cls_, s_, TR, VA)
                pr = predict(mdl, TEST)['fused_logits'].argmax(-1)
                accs.append(float((pr == yt).float().mean())); f1s.append(float(f1_score(yt.numpy(), pr.numpy(), average='macro')))
                preds[f'{nm_}_s{s_}'] = pr.numpy()
                if s_ == 42: MODELS_F[nm_] = mdl
                else: del mdl
            res['fusion'][nm_] = {'acc': accs, 'f1': f1s, 'acc_mean': float(np.mean(accs)), 'f1_mean': float(np.mean(f1s))}
            print(f"  fold {fold} {nm_}: acc {np.mean(accs)*100:.2f} (seeds {[round(a*100,2) for a in accs]})", flush=True)
        print(f"  fold {fold} text {res['pairs_text_acc']*100:.2f} | audio {res['pairs_audio_acc']*100:.2f} | late {res['pairs_late_acc']*100:.2f}", flush=True)

        # ---- incongruence: synthetic pairs (val for fitting, test for evaluation) ----
        seqm = MODELS_F['seq']
        va_all = np.isin(A_ACT, va_act)
        Xv = np.r_[alert_features(build_pairs('validation', ALL('validation'), va_all, 10, True), seqm),
                   alert_features(build_pairs('validation', ALL('validation'), va_all, 11, False), seqm)]
        yv = np.r_[np.zeros(len(ALL('validation'))), np.ones(len(ALL('validation')))]
        Xt = np.r_[alert_features(build_pairs('test', ALL('test'), te_mask, 20, True), seqm),
                   alert_features(build_pairs('test', ALL('test'), te_mask, 21, False), seqm)]
        ytt = np.r_[np.zeros(len(ALL('test'))), np.ones(len(ALL('test')))]
        lr_ = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xv, yv)
        sv, st = lr_.predict_proba(Xv)[:, 1], lr_.predict_proba(Xt)[:, 1]
        thr_l, thr_j = youden(sv, yv), youden(Xv[:, 0], yv)
        res['incongruence_synthetic'] = {
            'auc_jsd': float(roc_auc_score(ytt, Xt[:, 0])), 'auc_aux_jsd': float(roc_auc_score(ytt, Xt[:, 6])),
            'auc_learned': float(roc_auc_score(ytt, st)),
            'rule_mismatch_or_jsd05': rates(np.maximum(Xt[:, 5], (Xt[:, 0] >= 0.5).astype(float)), 0.5, ytt),
            'jsd_calibrated': {'thr': thr_j, **rates(Xt[:, 0], thr_j, ytt)},
            'learned_youden': {'thr': thr_l, **rates(st, thr_l, ytt)},
            'learned_coef': lr_[-1].coef_[0].tolist()}
        # ---- RAVDESS same-utterance test: real transcript + its own clip (fold test actors only) ----
        sel = np.nonzero(te_mask & np.isin(EMO, [1, 3, 4, 5, 6, 7]))[0]          # neutral = congruent; happy/sad/angry/fearful/disgust = incongruent
        SU = {'ts': 'sent', 'pairs': [(int(STMT[i]) - 1, int(i), int(A_CL[i])) for i in sel]}
        Xs = alert_features(SU, seqm); ys = (EMO[sel] != 1).astype(int)
        ss = lr_.predict_proba(Xs)[:, 1]
        res['same_utterance'] = {'n_congruent': int((ys == 0).sum()), 'n_incongruent': int((ys == 1).sum()),
            'auc_jsd': float(roc_auc_score(ys, Xs[:, 0])), 'auc_learned': float(roc_auc_score(ys, ss)),
            'rule_mismatch_or_jsd05': rates(np.maximum(Xs[:, 5], (Xs[:, 0] >= 0.5).astype(float)), 0.5, ys),
            'jsd_calibrated': rates(Xs[:, 0], thr_j, ys), 'learned_youden': rates(ss, thr_l, ys),
            'text_pred_on_transcripts': [CLUSTER_NAMES[int(i)] for i in T['sent']['probs'].argmax(-1)]}
        np.savez_compressed(ARR_DIR / f'fold{fold}.npz', **preds, Xt=Xt, yt_inc=ytt, st=st, Xs=Xs, ys=ys, ss=ss,
                            su_emo=EMO[sel], su_actor=A_ACT[sel], test_clip_actor=A_ACT, aprob=APROB.numpy())
        res['minutes'] = (time.time() - t_f) / 60
        CV['folds'][str(fold)] = res; write_cv()
        print(f"  fold {fold} JSD AUC {res['incongruence_synthetic']['auc_jsd']:.3f} | learned AUC {res['incongruence_synthetic']['auc_learned']:.3f} | "
              f"same-utterance JSD AUC {res['same_utterance']['auc_jsd']:.3f} | {res['minutes']:.0f} min", flush=True)
        del AF, AM, AEMB, APROB, MODELS_F, seqm; torch.cuda.empty_cache(); gc.collect()
    except Exception as e:
        traceback.print_exc(); CV['errors'].append({'fold': fold, 'error': repr(e)}); write_cv()
print('===== SUMMARY START ====='); print(json.dumps(CV, indent=1, default=_json_default)); print('===== SUMMARY END =====')